# AOI 3 Bologna MintPy Workflow

This notebook is wired to the HyP3 interferogram stack in `/mnt/data/aoi_3_bologna`.

Use it to:
- validate the interferogram inventory before running MintPy,
- run `smallbaselineApp.py` one step at a time from inside the Jupyter container,
- inspect the loaded stack, network, coherence, reference information, and time-series outputs.


In [ ]:
from pathlib import Path
import glob
import os
import shlex
import subprocess

import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, display

PROJECT_DIR = Path('/mnt/data/aoi_3_bologna')
WORK_DIR = PROJECT_DIR / 'mintpy'
FILTERED_VIEW_DIR = Path('/workspace/outputs/aoi_3_bologna_filtered')
RAW_CONFIG_FILE = Path('/workspace/configs/aoi_3_bologna_hyp3.cfg')
FILTERED_CONFIG_FILE = Path('/workspace/configs/aoi_3_bologna_hyp3_filtered.cfg')
CONFIG_FILE = RAW_CONFIG_FILE
PAIR_DIRS = sorted(p for p in PROJECT_DIR.iterdir() if p.is_dir() and p.name.startswith('S1'))

print(f'Project dir : {PROJECT_DIR}')
print(f'Work dir    : {WORK_DIR}')
print(f'Raw config  : {RAW_CONFIG_FILE}')
print(f'Filtered cfg: {FILTERED_CONFIG_FILE}')
print(f'Active cfg  : {CONFIG_FILE}')
print(f'Pair dirs   : {len(PAIR_DIRS)}')


## 1. Preflight Inventory

This checks whether the HyP3 interferogram folders are complete enough for MintPy loading.


In [ ]:
expected = [
    '_unw_phase_clipped.tif',
    '_corr_clipped.tif',
    '_dem_clipped.tif',
    '_lv_theta_clipped.tif',
    '_lv_phi_clipped.tif',
    '_water_mask_clipped.tif',
]

records = []
for pair_dir in PAIR_DIRS:
    row = {'pair_dir': pair_dir.name}
    missing = []
    for suffix in expected:
        present = any(pair_dir.glob(f'*{suffix}'))
        row[suffix] = present
        if not present:
            missing.append(suffix)
    row['missing'] = ', '.join(missing)
    records.append(row)

inventory = pd.DataFrame(records)
display(inventory.head())
print('Complete folders:', (inventory['missing'] == '').sum())
print('Folders with missing files:', (inventory['missing'] != '').sum())
display(inventory.loc[inventory['missing'] != '', ['pair_dir', 'missing']].head(20))


## 1b. Build a Filtered View (Recommended)

This creates a symlink-based view under `/workspace/outputs/aoi_3_bologna_filtered` that keeps only complete interferogram folders.

It does **not** modify the original dataset in `/mnt/data/aoi_3_bologna`.


In [ ]:
PREP_FILTERED_CMD = [
    'python3',
    '/workspace/scripts/06_prepare_hyp3_filtered_view.py',
    '--source', str(PROJECT_DIR),
    '--output', str(FILTERED_VIEW_DIR),
]
print(' '.join(map(str, PREP_FILTERED_CMD)))
# run_cmd(PREP_FILTERED_CMD)


In [ ]:
if FILTERED_VIEW_DIR.exists():
    manifest = FILTERED_VIEW_DIR / 'manifest.txt'
    links = sorted(p for p in FILTERED_VIEW_DIR.iterdir() if p.is_dir() or p.is_symlink())
    print('Filtered pair dirs:', len(links))
    if manifest.exists():
        print('\nLast 20 manifest lines:')
        print('\n'.join(manifest.read_text(errors='ignore').splitlines()[-20:]))
else:
    print('Filtered view not created yet. Run PREP_FILTERED_CMD first.')


In [ ]:
# Switch to the filtered config after the filtered view exists.
# CONFIG_FILE = FILTERED_CONFIG_FILE
# WORK_DIR = PROJECT_DIR / 'mintpy_filtered'
# print('Active cfg :', CONFIG_FILE)
# print('Work dir   :', WORK_DIR)


## 2. Active MintPy Configuration

This is the exact template that `smallbaselineApp.py` will read.


In [ ]:
print(CONFIG_FILE.read_text())


## 3. Step-by-Step Execution Helpers

These commands run **inside the Jupyter container**, so they call MintPy directly instead of using `docker compose` again.


In [ ]:
def run_cmd(cmd, cwd=None):
    print('+', ' '.join(shlex.quote(str(part)) for part in cmd))
    return subprocess.run([str(part) for part in cmd], cwd=cwd, text=True, check=False)

LOAD_DATA_CMD = [
    'smallbaselineApp.py', str(CONFIG_FILE), '--dir', str(WORK_DIR), '--dostep', 'load_data'
]
MODIFY_NETWORK_CMD = [
    'smallbaselineApp.py', str(CONFIG_FILE), '--dir', str(WORK_DIR), '--dostep', 'modify_network'
]
REFERENCE_POINT_CMD = [
    'smallbaselineApp.py', str(CONFIG_FILE), '--dir', str(WORK_DIR), '--dostep', 'reference_point'
]
FULL_RUN_CMD = [
    'smallbaselineApp.py', str(CONFIG_FILE), '--dir', str(WORK_DIR)
]

print('Load data       :', ' '.join(map(str, LOAD_DATA_CMD)))
print('Modify network  :', ' '.join(map(str, MODIFY_NETWORK_CMD)))
print('Reference point :', ' '.join(map(str, REFERENCE_POINT_CMD)))
print('Full run        :', ' '.join(map(str, FULL_RUN_CMD)))


In [ ]:
# Uncomment one command at a time when you are ready.
# run_cmd(LOAD_DATA_CMD)
# run_cmd(MODIFY_NETWORK_CMD)
# run_cmd(REFERENCE_POINT_CMD)
# run_cmd(FULL_RUN_CMD)


## 4. Inspect `load_data` Outputs

After `load_data`, MintPy should create `inputs/ifgramStack.h5` and the geometry files.


In [ ]:
stack_file = WORK_DIR / 'inputs' / 'ifgramStack.h5'
geom_radar = WORK_DIR / 'inputs' / 'geometryRadar.h5'

print('ifgramStack exists   :', stack_file.exists())
print('geometryRadar exists :', geom_radar.exists())

if stack_file.exists():
    with h5py.File(stack_file, 'r') as f:
        print('Datasets:')
        for key in f.keys():
            obj = f[key]
            shape = getattr(obj, 'shape', None)
            print(f'  - {key}: {shape}')

        if 'date' in f:
            dates = [d.decode() if isinstance(d, bytes) else str(d) for d in f['date'][:]]
            print(f'Acquisitions: {len(dates)}')
            print(f'Date range  : {dates[0]} -> {dates[-1]}')

        if 'date12' in f:
            date12 = [d.decode() if isinstance(d, bytes) else str(d) for d in f['date12'][:]]
            print(f'Interferograms: {len(date12)}')
            print('First 10 date12:', date12[:10])
else:
    print('Run LOAD_DATA_CMD first.')


## 5. Network View

This is useful after `load_data` and especially after `modify_network`.


In [ ]:
if stack_file.exists():
    run_cmd(['plot_network.py', str(stack_file), '--nodisplay', '--save'], cwd=WORK_DIR)
    candidates = sorted(WORK_DIR.glob('network*.png')) + sorted(WORK_DIR.glob('Network*.png'))
    if candidates:
        print('Displaying:', candidates[-1])
        display(Image(filename=str(candidates[-1])))
    else:
        print('No network PNG found yet.')
else:
    print('Run LOAD_DATA_CMD first.')


## 6. Watch Logs and Generated Files

This is the closest thing to a live dashboard in Jupyter. Re-run the cell while MintPy is working.


In [ ]:
log_dir = WORK_DIR / 'logs'
print('Work dir contents:')
for item in sorted(WORK_DIR.glob('*'))[:40]:
    print(' -', item.name)

if log_dir.exists():
    print('\nRecent step logs:')
    for log_file in sorted(log_dir.glob('*.log'))[-10:]:
        print(' -', log_file.name)
else:
    print('\nNo step log directory yet.')

pipeline_log = WORK_DIR / 'pipeline.log'
if pipeline_log.exists():
    print('\nLast 40 lines of pipeline.log:')
    print('\n'.join(pipeline_log.read_text(errors='ignore').splitlines()[-40:]))


## 7. Reference Information and Coherence

These become useful after `reference_point`, `invert_network`, and later steps.


In [ ]:
coh_file = WORK_DIR / 'temporalCoherence.h5'
avg_file = WORK_DIR / 'avgSpatialCoh.h5'
ts_file = WORK_DIR / 'timeseries.h5'

for product in [avg_file, coh_file, ts_file]:
    print(product.name, product.exists())

if coh_file.exists():
    with h5py.File(coh_file, 'r') as f:
        key = list(f.keys())[0]
        coh = f[key][:]
    plt.figure(figsize=(8, 6))
    plt.imshow(coh, cmap='gray', vmin=0, vmax=1)
    plt.colorbar(label='Temporal coherence')
    plt.title('Temporal Coherence')
    plt.tight_layout()
    plt.show()

if ts_file.exists():
    with h5py.File(ts_file, 'r') as f:
        attrs = dict(f.attrs)
    for key in ['REF_X', 'REF_Y', 'REF_LAT', 'REF_LON', 'REF_DATE']:
        if key in attrs:
            print(f'{key}: {attrs[key]}')


## 8. Velocity and Time-Series Inspection

These cells are useful after `velocity` has been created.


In [ ]:
velocity_file = WORK_DIR / 'velocity.h5'

if velocity_file.exists():
    with h5py.File(velocity_file, 'r') as f:
        vel = f['velocity'][:] * 100.0
    plt.figure(figsize=(8, 6))
    plt.imshow(vel, cmap='RdBu_r')
    plt.colorbar(label='cm/yr')
    plt.title('Velocity')
    plt.tight_layout()
    plt.show()
else:
    print('velocity.h5 not available yet.')

if ts_file.exists():
    with h5py.File(ts_file, 'r') as f:
        ts = f['timeseries'][:]
        dates = [d.decode() if isinstance(d, bytes) else str(d) for d in f['date'][:]]
        attrs = dict(f.attrs)
    ref_y = int(attrs.get('REF_Y', ts.shape[1] // 2))
    ref_x = int(attrs.get('REF_X', ts.shape[2] // 2))
    series = ts[:, ref_y, ref_x] * 100.0
    plt.figure(figsize=(10, 4))
    plt.plot(dates, series, marker='o', linewidth=1)
    plt.xticks(rotation=60)
    plt.ylabel('cm')
    plt.title(f'Time series at reference pixel ({ref_y}, {ref_x})')
    plt.tight_layout()
    plt.show()


## 9. Useful MintPy CLI Checks

Run these as needed after files appear in the work directory.


In [ ]:
# Examples:
# run_cmd(['info.py', str(stack_file)])
# run_cmd(['info.py', str(ts_file)])
# run_cmd(['info.py', str(velocity_file)])
# run_cmd(['view.py', str(velocity_file), 'velocity', '--nodisplay', '--save'], cwd=WORK_DIR)
